# NexaTel Customer Churn Analytics project

## Phase 1 — Data Cleaning & Quality Check  

## Load All CSVs

In [44]:
import pandas as pd
from pathlib import Path

# The data folder is beside this notebook
data_folder = Path("data")

print("Notebook working folder:", Path.cwd())
print("Data folder found:", data_folder.is_dir())

# Find and load CSV files
csv_files = sorted(data_folder.glob("*.csv"))

tables = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

print("Total files loaded:", len(tables))
print("Total rows:", sum(len(df) for df in tables.values()))

Notebook working folder: C:\Users\Anushka Tyagi\NEXATEL_PROJECT
Data folder found: True
Total files loaded: 24
Total rows: 756952


## Basic Profile (Row Count, Column Count, Data Types)

In [45]:
summary = pd.DataFrame([
    {
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in tables.items()
])

display(summary)

print("Total tables:", len(summary))
print("Total rows:", summary["Rows"].sum())

,Table,Rows,Columns
0,billing,115313,9
1,cities,68,8
2,complaints,16584,7
3,contracts,10607,9
4,customer_feedback,28499,7
5,customers,19076,25
6,data_quality_issue_log,20,3
7,devices,19000,8
8,employees,800,9
9,marketing_campaigns,40,9


Total tables: 24
Total rows: 756952


In [46]:
for name, df in tables.items():
    print(f"\nTABLE: {name}")

    # Show the first 3 records
    display(df.head(3))

    # Show column names, non-null counts, and data types
    df.info()

    print("-" * 60)


TABLE: billing


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
0,INV00000001,C0000002,31-03-2026,03-2026,1490.77,268.34,1759.11,15-04-2026,Paid
1,INV00000002,C0000002,01-03-2026,03-2026,1396.28,251.33,1647.61,16-03-2026,Overdue
2,INV00000003,C0000002,30-01-2026,01-2026,1427.24,256.90,1684.14,14-02-2026,Paid


<class 'pandas.DataFrame'>
RangeIndex: 115313 entries, 0 to 115312
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   invoice_id            115313 non-null  str    
 1   customer_id           115313 non-null  str    
 2   billing_date          115313 non-null  str    
 3   billing_period_month  115313 non-null  str    
 4   base_amount_inr       115313 non-null  float64
 5   gst_amount_inr        115313 non-null  float64
 6   total_amount_inr      115313 non-null  float64
 7   due_date              115313 non-null  str    
 8   payment_status        115313 non-null  str    
dtypes: float64(3), str(6)
memory usage: 13.5 MB
------------------------------------------------------------

TABLE: cities


,city_id,city_name,state_id,state_name,zone,region_id,tier,pin_prefix
0,CT0001,New Delhi,ST001,Delhi,North,RG01,Metro,110
1,CT0002,Mumbai,ST010,Maharashtra,West,RG02,Metro,400
2,CT0003,Bengaluru,ST015,Karnataka,South,RG03,Metro,560


<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   city_id     68 non-null     str  
 1   city_name   68 non-null     str  
 2   state_id    68 non-null     str  
 3   state_name  68 non-null     str  
 4   zone        68 non-null     str  
 5   region_id   68 non-null     str  
 6   tier        68 non-null     str  
 7   pin_prefix  68 non-null     int64
dtypes: int64(1), str(7)
memory usage: 7.2 KB
------------------------------------------------------------

TABLE: complaints


,complaint_id,customer_id,complaint_date,complaint_type,severity,status,resolution_date
0,CMP00000001,C0000001,30-06-2020,Data Speed,Major,Unresolved,NaN
1,CMP00000002,C0000002,01-12-2015,Network Coverage,Moderate,Resolved,02-12-2015
2,CMP00000003,C0000003,09-10-2014,Wrongful Charge,Major,Unresolved,NaN


... (39 outputs left out)

,usage_id,customer_id,usage_month,sms_sent,sms_received
0,US00000001,C0000001,03-2026,19,20
1,US00000002,C0000001,02-2026,32,18
2,US00000003,C0000001,01-2026,21,28


<class 'pandas.DataFrame'>
RangeIndex: 57000 entries, 0 to 56999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   usage_id      57000 non-null  str  
 1   customer_id   57000 non-null  str  
 2   usage_month   57000 non-null  str  
 3   sms_sent      57000 non-null  int64
 4   sms_received  57000 non-null  int64
dtypes: int64(2), str(3)
memory usage: 3.5 MB
------------------------------------------------------------

TABLE: usage_voice


,usage_id,customer_id,usage_month,voice_minutes_used,outgoing_calls,incoming_calls
0,UV00000001,C0000001,03-2026,439,356,255
1,UV00000002,C0000001,02-2026,705,32,694
2,UV00000003,C0000001,01-2026,206,339,194


<class 'pandas.DataFrame'>
RangeIndex: 57000 entries, 0 to 56999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   usage_id            57000 non-null  str  
 1   customer_id         57000 non-null  str  
 2   usage_month         57000 non-null  str  
 3   voice_minutes_used  57000 non-null  int64
 4   outgoing_calls      57000 non-null  int64
 5   incoming_calls      57000 non-null  int64
dtypes: int64(3), str(3)
memory usage: 4.0 MB
------------------------------------------------------------


## Missing Values

In [47]:
missing_records = []

for name, df in tables.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_records.append({
                "Table": name,
                "Column": column,
                "Missing Count": missing_count,
                "Missing %": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_report = pd.DataFrame(
    missing_records,
    columns=["Table", "Column", "Missing Count", "Missing %"]
)

display(missing_report)

,Table,Column,Missing Count,Missing %
0,complaints,resolution_date,7348,44.31
1,customers,email,745,3.91
2,customers,address,380,1.99
3,customers,annual_income_inr,954,5.00
4,customers,churn_date,15560,81.57
5,subscriptions,plan_id,41,0.20
6,subscriptions,end_date,17017,82.92
7,support_tickets,resolution_hours,13366,42.45
8,support_tickets,resolution_date,13366,42.45


## Duplicate Rows

In [48]:
duplicate_report = pd.DataFrame([
    {
        "Table": name,
        "Duplicate Rows": df.duplicated().sum()
    }
    for name, df in tables.items()
])

display(duplicate_report)

print(
    "Total duplicate rows:",
    duplicate_report["Duplicate Rows"].sum()
)

,Table,Duplicate Rows
0,billing,344
1,cities,0
2,complaints,0
3,contracts,0
4,customer_feedback,0
5,customers,76
6,data_quality_issue_log,0
7,devices,0
8,employees,0
9,marketing_campaigns,0


Total duplicate rows: 696


## Remove Duplicate Rows

In [49]:
# Create working copies and remove exact duplicate rows
clean_tables = {
    name: df.drop_duplicates().reset_index(drop=True).copy()
    for name, df in tables.items()
}

# Record how many rows were removed
cleaning_summary = pd.DataFrame([
    {
        "Table": name,
        "Original Rows": len(tables[name]),
        "Remaining Rows": len(clean_tables[name]),
        "Removed": len(tables[name]) - len(clean_tables[name])
    }
    for name in tables
])

display(cleaning_summary[cleaning_summary["Removed"] > 0])

print("Total duplicates removed:", cleaning_summary["Removed"].sum())
print("Total remaining rows:", sum(len(df) for df in clean_tables.values()))

,Table,Original Rows,Remaining Rows,Removed
0,billing,115313,114969,344
5,customers,19076,19000,76
11,payments,92362,92086,276


Total duplicates removed: 696
Total remaining rows: 756256


In [50]:
key_checks = {
    "customers": "customer_id",
    "billing": "invoice_id"
}

key_report = []

for table_name, key in key_checks.items():
    df = clean_tables[table_name]

    key_report.append({
        "Table": table_name,
        "Key Column": key,
        "Missing IDs": df[key].isna().sum(),
        "Extra Repeated IDs": df[key].dropna().duplicated().sum()
    })

display(pd.DataFrame(key_report))

,Table,Key Column,Missing IDs,Extra Repeated IDs
0,customers,customer_id,0,0
1,billing,invoice_id,0,0


## Check Date Formats

In [51]:
date_check = []

for name, df in clean_tables.items():
    for column in df.columns:
        if "date" in column.lower():
            date_check.append({
                "Table": name,
                "Column": column,
                "Data Type": str(df[column].dtype),
                "Sample Values": df[column].dropna().head(3).tolist()
            })

display(pd.DataFrame(date_check))

,Table,Column,Data Type,Sample Values
0,billing,billing_date,str,"[31-03-2026, 01-03-2026, 30-01-2026]"
1,billing,due_date,str,"[15-04-2026, 16-03-2026, 14-02-2026]"
2,complaints,complaint_date,str,"[30-06-2020, 01-12-2015, 09-10-2014]"
3,complaints,resolution_date,str,"[02-12-2015, 13-03-2022, 06-10-2021]"
4,contracts,start_date,str,"[01-01-2014, 11-11-2015, 18-10-2014]"
5,contracts,end_date,str,"[22-12-2015, 04-05-2017, 13-10-2015]"
6,customer_feedback,feedback_date,str,"[13-03-2025, 01-08-2019, 10-08-2019]"
7,customers,date_of_birth,str,"[31-03-1988, 31-03-1974, 31-03-1978]"
8,customers,acquisition_date,str,"[15-01-2026, 01-01-2014, 11-11-2015]"
9,customers,churn_date,str,"[22-03-2026, 22-12-2015, 22-10-2023]"


In [52]:
date_conversion_report = []
invalid_date_records = []

for name, df in clean_tables.items():
    for column in df.columns:
        if "date" in column.lower():

            # Skip columns already converted
            if pd.api.types.is_datetime64_any_dtype(df[column]):
                continue

            original = df[column].copy()

            converted = pd.to_datetime(
                original,
                format="%d-%m-%Y",
                errors="coerce"
            )

            failed = original.notna() & converted.isna()

            # Preserve failed values for investigation
            for row_index in df.index[failed]:
                invalid_date_records.append({
                    "Table": name,
                    "Column": column,
                    "Row": row_index,
                    "Original Value": original.loc[row_index]
                })

            df[column] = converted

            date_conversion_report.append({
                "Table": name,
                "Column": column,
                "Previously Missing": original.isna().sum(),
                "Failed Conversion": failed.sum(),
                "New Type": str(df[column].dtype)
            })

display(pd.DataFrame(date_conversion_report))

invalid_dates = pd.DataFrame(invalid_date_records)

print("Total failed date values:", len(invalid_dates))

if not invalid_dates.empty:
    display(invalid_dates.head(20))

,Table,Column,Previously Missing,Failed Conversion,New Type
0,billing,billing_date,0,0,datetime64[us]
1,billing,due_date,0,0,datetime64[us]
2,complaints,complaint_date,0,0,datetime64[us]
3,complaints,resolution_date,7348,0,datetime64[us]
4,contracts,start_date,0,0,datetime64[us]
5,contracts,end_date,0,0,datetime64[us]
6,customer_feedback,feedback_date,0,0,datetime64[us]
7,customers,date_of_birth,0,0,datetime64[us]
8,customers,acquisition_date,0,380,datetime64[us]
9,customers,churn_date,15494,0,datetime64[us]


Total failed date values: 380


,Table,Column,Row,Original Value
0,customers,acquisition_date,3,2014/10/18
1,customers,acquisition_date,33,2023/04/16
2,customers,acquisition_date,59,2021/06/13
3,customers,acquisition_date,74,2024/12/13
4,customers,acquisition_date,88,2021/10/31
5,customers,acquisition_date,94,2018/06/01
6,customers,acquisition_date,151,2019/07/28
7,customers,acquisition_date,308,2016/09/03
8,customers,acquisition_date,340,2014/05/30
9,customers,acquisition_date,347,2014/01/01


In [53]:
# Select the acquisition dates that failed conversion
failed = invalid_dates[
    (invalid_dates["Table"] == "customers") &
    (invalid_dates["Column"] == "acquisition_date")
].copy()

# Convert using the second date format
failed["Recovered Date"] = pd.to_datetime(
    failed["Original Value"],
    format="%Y/%m/%d",
    errors="coerce"
)

# Restore successfully converted dates to their original rows
recovered = failed.dropna(subset=["Recovered Date"])

for _, record in recovered.iterrows():
    clean_tables["customers"].at[
        record["Row"], "acquisition_date"
    ] = record["Recovered Date"]

print("Dates recovered:", len(recovered))
print("Still failed:", failed["Recovered Date"].isna().sum())
print(
    "Missing acquisition dates:",
    clean_tables["customers"]["acquisition_date"].isna().sum()
)

Dates recovered: 380
Still failed: 0
Missing acquisition dates: 0


In [54]:
numeric_check = []

for name, df in clean_tables.items():
    for column in df.columns:
        if any(word in column.lower()
               for word in ["amount", "charge", "income", "data_gb"]):

            numeric_check.append({
                "Table": name,
                "Column": column,
                "Data Type": str(df[column].dtype),
                "Sample Values": df[column].dropna().head(3).tolist()
            })

display(pd.DataFrame(numeric_check))

,Table,Column,Data Type,Sample Values
0,billing,base_amount_inr,float64,"[1490.77, 1396.28, 1427.24]"
1,billing,gst_amount_inr,float64,"[268.34, 251.33, 256.9]"
2,billing,total_amount_inr,float64,"[1759.11, 1647.61, 1684.14]"
3,customers,annual_income_inr,float64,"[791058.0, 1385499.0, 394773.0]"
4,payments,amount_inr,float64,"[1759.11, 1647.61, 1684.14]"
5,plans,monthly_charge,int64,"[199, 299, 399]"
6,recharges,recharge_id,str,"[RC00000001, RC00000002, RC00000003]"
7,recharges,recharge_date,datetime64[us],"[2026-01-18 00:00:00, 2026-02-07 00:00:00, 202..."
8,recharges,amount_inr,float64,"[303.9, 282.6, 313.5]"
9,subscriptions,monthly_charge_inr,float64,"[299.0, 1499.0, 4166.0]"


In [55]:
# 1. Check negative invoice totals
billing = clean_tables["billing"]

negative_bills = billing[billing["total_amount_inr"] < 0]

print("Negative invoice totals:", len(negative_bills))
display(negative_bills.head())

# 2. Flag unusually high or low usage using the IQR method
usage = clean_tables["usage_data"]
values = usage["data_gb_used"]

q1 = values.quantile(0.25)
q3 = values.quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

usage_outliers = usage[
    (values < lower_limit) | (values > upper_limit)
]

print("Negative usage values:", (values < 0).sum())
print("Lower IQR limit:", round(lower_limit, 2))
print("Upper IQR limit:", round(upper_limit, 2))
print("Usage outliers:", len(usage_outliers))

display(
    usage_outliers.sort_values(
        "data_gb_used", ascending=False
    ).head(10)
)

Negative invoice totals: 229


,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
352,INV00000353,C0000074,2025-12-01,12-2025,611.62,110.09,-721.71,2025-12-16,Paid
559,INV00000560,C0000102,2026-03-01,03-2026,639.46,115.10,-754.56,2026-03-16,Paid
716,INV00000717,C0000124,2025-09-02,09-2025,1324.07,238.33,-1562.40,2025-09-17,Overdue
867,INV00000868,C0000156,2025-10-02,10-2025,1408.69,253.56,-1662.25,2025-10-17,Paid
1030,INV00001031,C0000192,2026-03-31,03-2026,531.10,95.60,-626.70,2026-04-15,Unpaid


Negative usage values: 0
Lower IQR limit: -20.47
Upper IQR limit: 64.49
Usage outliers: 4480


,usage_id,customer_id,usage_month,data_gb_used,data_sessions
15077,UD00015078,C0005026,01-2026,22172.700708,31
23089,UD00023090,C0007697,02-2026,21960.635745,489
10054,UD00010055,C0003352,02-2026,18348.446462,369
34859,UD00034860,C0011620,01-2026,17377.228312,765
26592,UD00026593,C0008865,03-2026,15322.701632,268
29837,UD00029838,C0009946,01-2026,13886.727541,1105
53991,UD00053992,C0017998,03-2026,13344.522727,862
56096,UD00056097,C0018699,01-2026,13154.906387,317
45208,UD00045209,C0015070,02-2026,12747.455634,929
19929,UD00019930,C0006644,03-2026,12581.380386,496


In [56]:
# Review the supplied data-quality issue log
with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(tables["data_quality_issue_log"])

# Investigate negative invoice totals
negative_bill_check = negative_bills.copy()

negative_bill_check["expected_total"] = (
    negative_bill_check["base_amount_inr"]
    + negative_bill_check["gst_amount_inr"]
).round(2)

negative_bill_check["matches_if_sign_reversed"] = (
    (
        negative_bill_check["total_amount_inr"].abs()
        - negative_bill_check["expected_total"]
    ).abs() < 0.02
)

print("Negative invoices:", len(negative_bill_check))
print(
    "Match base + GST after reversing sign:",
    negative_bill_check["matches_if_sign_reversed"].sum()
)

display(
    negative_bill_check[
        [
            "invoice_id",
            "total_amount_inr",
            "expected_total",
            "matches_if_sign_reversed"
        ]
    ].head(10)
)

,table,issue_type,rows_affected
0,customers,missing_email,760
1,customers,missing_address,380
2,customers,missing_income,950
3,customers,invalid_phone,228
4,customers,invalid_email,285
5,customers,incorrect_pin,190
6,customers,misspelled_city,380
7,customers,inconsistent_capitalisation,570
8,customers,impossible_age,38
9,customers,mixed_date_format,380


Negative invoices: 229
Match base + GST after reversing sign: 228


,invoice_id,total_amount_inr,expected_total,matches_if_sign_reversed
352,INV00000353,-721.71,721.71,True
559,INV00000560,-754.56,754.56,True
716,INV00000717,-1562.40,1562.40,True
867,INV00000868,-1662.25,1662.25,True
1030,INV00001031,-626.70,626.70,True
1900,INV00001901,-1793.40,1793.40,True
3810,INV00003811,-4612.79,4612.79,True
4479,INV00004480,-1787.97,1787.97,True
4634,INV00004635,-558.22,558.22,True
5094,INV00005095,-926.08,926.08,True


In [57]:
billing = clean_tables["billing"]

# Preserve original totals for comparison
if "original_total_amount_inr" not in billing.columns:
    billing["original_total_amount_inr"] = billing["total_amount_inr"]

expected_total = (
    billing["base_amount_inr"] + billing["gst_amount_inr"]
).round(2)

# Correct only negative totals whose magnitude matches base + GST
fix_mask = (
    (billing["total_amount_inr"] < 0)
    & (expected_total > 0)
    & (
        (billing["total_amount_inr"].abs() - expected_total).abs()
        < 0.02
    )
)

billing.loc[fix_mask, "total_amount_inr"] = (
    billing.loc[fix_mask, "total_amount_inr"].abs()
)

print("Negative totals corrected:", fix_mask.sum())

# Inspect the remaining negative invoice
remaining_negative = billing[
    billing["total_amount_inr"] < 0
].copy()

remaining_negative["base_plus_gst"] = (
    remaining_negative["base_amount_inr"]
    + remaining_negative["gst_amount_inr"]
).round(2)

print("Negative totals remaining:", len(remaining_negative))

display(
    remaining_negative[
        [
            "invoice_id",
            "base_amount_inr",
            "gst_amount_inr",
            "total_amount_inr",
            "base_plus_gst"
        ]
    ]
)

Negative totals corrected: 228
Negative totals remaining: 1


,invoice_id,base_amount_inr,gst_amount_inr,total_amount_inr,base_plus_gst
105595,INV00105596,464.91,51.72,-548.59,516.63


In [58]:
billing = clean_tables["billing"]

# Working assumption to validate against the dataset
gst_rate = 0.18

gst_check = billing[
    [
        "invoice_id",
        "base_amount_inr",
        "gst_amount_inr",
        "total_amount_inr"
    ]
].copy()

gst_check["expected_gst"] = (
    gst_check["base_amount_inr"] * gst_rate
).round(2)

gst_check["gst_difference"] = (
    gst_check["gst_amount_inr"] - gst_check["expected_gst"]
).round(2)

gst_errors = gst_check[
    gst_check["gst_difference"].abs() > 0.01
]

print("GST mismatches using 18%:", len(gst_errors))
print("Incorrect GST records listed in issue log: 1149")

display(gst_errors.head(10))

GST mismatches using 18%: 1149
Incorrect GST records listed in issue log: 1149


,invoice_id,base_amount_inr,gst_amount_inr,total_amount_inr,expected_gst,gst_difference
155,INV00000156,1065.44,121.51,1257.22,191.78,-70.27
209,INV00000210,1011.24,141.37,1193.26,182.02,-40.65
255,INV00000256,504.42,35.10,595.22,90.80,-55.70
463,INV00000464,1059.72,145.90,1250.47,190.75,-44.85
473,INV00000474,90.39,16.76,106.66,16.27,0.49
674,INV00000675,8071.35,1873.71,9524.19,1452.84,420.87
698,INV00000699,1005.08,238.05,1185.99,180.91,57.14
1097,INV00001098,665.77,34.36,785.61,119.84,-85.48
1168,INV00001169,618.68,111.44,730.04,111.36,0.08
1352,INV00001353,518.85,123.12,612.24,93.39,29.73


In [59]:
billing = clean_tables["billing"]

# Preserve original values for the quality report
if "original_gst_amount_inr" not in billing.columns:
    billing["original_gst_amount_inr"] = billing["gst_amount_inr"]

if "original_total_amount_inr" not in billing.columns:
    billing["original_total_amount_inr"] = billing["total_amount_inr"]

# Correct GST mismatches
expected_gst = (billing["base_amount_inr"] * 0.18).round(2)

gst_mask = (
    (billing["gst_amount_inr"] - expected_gst)
    .round(2).abs() > 0.01
)

billing.loc[gst_mask, "gst_amount_inr"] = expected_gst[gst_mask]

# Correct remaining negative totals only when their magnitude
# matches base + corrected GST
expected_total = (
    billing["base_amount_inr"] + billing["gst_amount_inr"]
).round(2)

negative_mask = (
    (billing["total_amount_inr"] < 0)
    & (expected_total > 0)
    & (
        (billing["total_amount_inr"].abs() - expected_total)
        .round(2).abs() <= 0.01
    )
)

billing.loc[negative_mask, "total_amount_inr"] = (
    expected_total[negative_mask]
)

# Validate the results
print("GST values corrected:", gst_mask.sum())
print("Negative totals corrected:", negative_mask.sum())

print(
    "GST mismatches remaining:",
    (
        (billing["gst_amount_inr"] - expected_gst)
        .round(2).abs() > 0.01
    ).sum()
)

print(
    "Negative totals remaining:",
    (billing["total_amount_inr"] < 0).sum()
)

print(
    "Totals differing from base + GST:",
    (
        (billing["total_amount_inr"] - expected_total)
        .round(2).abs() > 0.01
    ).sum()
)

GST values corrected: 1149
Negative totals corrected: 1
GST mismatches remaining: 0
Negative totals remaining: 0
Totals differing from base + GST: 0


In [60]:
billing = clean_tables["billing"]

print("Earliest billing date:", billing["billing_date"].min())
print("Latest billing date:", billing["billing_date"].max())

print("\nBilling records by year and month:")
display(
    billing["billing_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename("Invoice Count")
    .to_frame()
)

print("\n10 invoices with the latest billing dates:")
display(
    billing.nlargest(10, "billing_date")[
        ["invoice_id", "billing_date", "billing_period_month", "due_date"]
    ]
)

Earliest billing date: 2014-02-01 00:00:00
Latest billing date: 2027-04-28 00:00:00

Billing records by year and month:


,Invoice Count
billing_date,
2014-02,12
2014-03,15
2014-04,12
2014-05,13
2014-06,14
...,...
2026-12,20
2027-01,18
2027-02,20



10 invoices with the latest billing dates:


,invoice_id,billing_date,billing_period_month,due_date
19201,INV00019202,2027-04-28,03-2026,2026-03-16
9101,INV00009102,2027-04-25,08-2025,2025-08-18
54756,INV00054757,2027-04-22,12-2020,2021-01-03
8746,INV00008747,2027-04-19,02-2026,2026-02-17
52555,INV00052556,2027-04-12,10-2025,2025-10-17
1584,INV00001585,2027-04-11,05-2025,2025-05-20
78706,INV00078707,2027-04-10,05-2025,2025-05-20
91985,INV00091986,2027-04-10,08-2025,2025-08-18
31875,INV00031876,2027-04-08,10-2025,2025-10-17
33743,INV00033744,2027-04-02,12-2025,2025-12-16


In [61]:
billing = clean_tables["billing"]

# Calculate the number of days allowed for payment
date_gap = (billing["due_date"] - billing["billing_date"]).dt.days

print("Most common gaps between billing date and due date:")
display(
    date_gap.value_counts()
    .head(10)
    .rename_axis("Gap in Days")
    .reset_index(name="Invoice Count")
)

# Flag invoices dated after their due date
date_issue_mask = billing["billing_date"] > billing["due_date"]

print("Invoices dated after their due date:", date_issue_mask.sum())

display(
    billing.loc[
        date_issue_mask,
        ["invoice_id", "billing_date", "billing_period_month", "due_date"]
    ].head(10)
)

Most common gaps between billing date and due date:


,Gap in Days,Invoice Count
0,15,114740
1,-333,4
2,-482,3
3,-369,3
4,-320,3
5,-382,3
6,-208,2
7,-350,2
8,-122,2
9,-277,2


Invoices dated after their due date: 229


,invoice_id,billing_date,billing_period_month,due_date
148,INV00000149,2026-08-11,12-2025,2026-01-15
1584,INV00001585,2027-04-11,05-2025,2025-05-20
1990,INV00001991,2026-09-27,06-2025,2025-06-19
2829,INV00002830,2027-01-30,01-2026,2026-02-14
3010,INV00003011,2027-02-01,03-2026,2026-04-15
3059,INV00003060,2026-12-11,03-2026,2026-04-15
3777,INV00003778,2026-08-02,10-2025,2025-10-17
4024,INV00004025,2026-06-30,07-2025,2025-07-19
5540,INV00005541,2026-06-16,01-2026,2026-02-14
6422,INV00006423,2026-08-15,12-2025,2026-01-15


In [62]:
billing = clean_tables["billing"]

# Preserve original dates for the audit trail
if "original_billing_date" not in billing.columns:
    billing["original_billing_date"] = billing["billing_date"]

# Identify inconsistent dates
date_fix_mask = billing["billing_date"] > billing["due_date"]

# Keep a permanent flag for inferred corrections
if "billing_date_inferred" not in billing.columns:
    billing["billing_date_inferred"] = False

billing.loc[date_fix_mask, "billing_date_inferred"] = True

# Apply the observed 15-day payment-period rule
billing.loc[date_fix_mask, "billing_date"] = (
    billing.loc[date_fix_mask, "due_date"]
    - pd.Timedelta(days=15)
)

print("Billing dates corrected:", date_fix_mask.sum())

print(
    "Billing dates after due date:",
    (billing["billing_date"] > billing["due_date"]).sum()
)

display(
    billing.loc[
        billing["billing_date_inferred"],
        ["invoice_id", "original_billing_date",
         "billing_date", "billing_period_month", "due_date"]
    ].head(10)
)

Billing dates corrected: 229
Billing dates after due date: 0


,invoice_id,original_billing_date,billing_date,billing_period_month,due_date
148,INV00000149,2026-08-11,2025-12-31,12-2025,2026-01-15
1584,INV00001585,2027-04-11,2025-05-05,05-2025,2025-05-20
1990,INV00001991,2026-09-27,2025-06-04,06-2025,2025-06-19
2829,INV00002830,2027-01-30,2026-01-30,01-2026,2026-02-14
3010,INV00003011,2027-02-01,2026-03-31,03-2026,2026-04-15
3059,INV00003060,2026-12-11,2026-03-31,03-2026,2026-04-15
3777,INV00003778,2026-08-02,2025-10-02,10-2025,2025-10-17
4024,INV00004025,2026-06-30,2025-07-04,07-2025,2025-07-19
5540,INV00005541,2026-06-16,2026-01-30,01-2026,2026-02-14
6422,INV00006423,2026-08-15,2025-12-31,12-2025,2026-01-15


In [63]:
payments = clean_tables["payments"]
billing = clean_tables["billing"]

# Separate missing IDs from IDs that do not exist in billing
missing_invoice_mask = payments["invoice_id"].isna()

broken_invoice_mask = (
    payments["invoice_id"].notna()
    & ~payments["invoice_id"].isin(billing["invoice_id"])
)

# Preserve a flag for later review
payments["invalid_invoice_reference"] = (
    missing_invoice_mask | broken_invoice_mask
)

print("Payments with missing invoice IDs:", missing_invoice_mask.sum())
print("Payments referencing nonexistent invoices:", broken_invoice_mask.sum())

display(
    payments.loc[broken_invoice_mask].head(10)
)

Payments with missing invoice IDs: 0
Payments referencing nonexistent invoices: 184


,payment_id,invoice_id,customer_id,payment_date,amount_inr,payment_method,payment_status,invalid_invoice_reference
874,PAY00000875,INV94560686,C0000199,2026-01-11,1444.45,Credit Card,Success,True
1007,PAY00001008,INV96995702,C0000232,2025-12-27,723.82,Credit Card,Success,True
1474,PAY00001475,INV99552429,C0000319,2025-05-24,1439.82,Credit Card,Success,True
2112,PAY00002113,INV98863464,C0000473,2025-07-08,746.33,UPI,Success,True
2117,PAY00002118,INV92113763,C0000477,2025-11-07,1536.96,Cash,Success,True
2380,PAY00002381,INV92607579,C0000533,2025-09-06,817.30,Wallet,Success,True
2759,PAY00002760,INV98698649,C0000628,2025-08-11,850.38,Auto Debit,Success,True
3251,PAY00003252,INV98048665,C0000733,2026-02-08,343.09,Net Banking,Success,True
3589,PAY00003590,INV97878268,C0000827,2026-02-05,694.26,IMPS,Success,True
3937,PAY00003938,INV97615652,C0000892,2024-05-10,1483.26,Net Banking,Success,True


In [64]:
subscriptions = clean_tables["subscriptions"]
plans = clean_tables["plans"]

# Check missing and nonexistent plan IDs separately
missing_plan_mask = subscriptions["plan_id"].isna()

broken_plan_mask = (
    subscriptions["plan_id"].notna()
    & ~subscriptions["plan_id"].isin(plans["plan_id"])
)

subscriptions["invalid_plan_reference"] = (
    missing_plan_mask | broken_plan_mask
)

print("Subscriptions with missing plan IDs:", missing_plan_mask.sum())
print("Subscriptions with nonexistent plan IDs:", broken_plan_mask.sum())

print("\nSubscription status counts:")
display(
    subscriptions["subscription_status"]
    .value_counts(dropna=False)
    .rename_axis("Subscription Status")
    .reset_index(name="Count")
)

print("\nSample subscriptions with missing plan IDs:")
display(subscriptions.loc[missing_plan_mask].head(5))

Subscriptions with missing plan IDs: 41
Subscriptions with nonexistent plan IDs: 0

Subscription status counts:


,Subscription Status,Count
0,Active,16930
1,Terminated,3491
2,UNKNOWN_STATE,102



Sample subscriptions with missing plan IDs:


,subscription_id,customer_id,plan_id,start_date,end_date,billing_cycle,monthly_charge_inr,subscription_status,invalid_plan_reference
17,S0000018,C0000018,NaN,2014-02-05,2025-02-10,Monthly,299.0,Terminated,True
203,S0000204,C0000204,NaN,2022-01-23,NaT,Monthly,449.0,Active,True
977,S0000978,C0000978,NaN,2024-05-03,NaT,Monthly,599.0,Active,True
1070,S0001071,C0001071,NaN,2016-05-06,NaT,Monthly,1499.0,Active,True
1352,S0001353,C0001353,NaN,2018-08-26,NaT,Monthly,599.0,Active,True


In [65]:
subscriptions = clean_tables["subscriptions"]

# Compare subscription status with the presence of an end date
status_date_check = pd.crosstab(
    subscriptions["subscription_status"],
    subscriptions["end_date"].notna()
).rename(columns={
    False: "No End Date",
    True: "Has End Date"
})

display(status_date_check)

# Flag the invalid statuses
subscriptions["invalid_subscription_status"] = (
    subscriptions["subscription_status"] == "UNKNOWN_STATE"
)

print("\nSample subscriptions with invalid status:")
display(
    subscriptions.loc[
        subscriptions["invalid_subscription_status"],
        ["subscription_id", "customer_id",
         "start_date", "end_date", "subscription_status"]
    ].head(10)
)

end_date,No End Date,Has End Date
subscription_status,,
Active,16930,0
Terminated,0,3491
UNKNOWN_STATE,87,15



Sample subscriptions with invalid status:


,subscription_id,customer_id,start_date,end_date,subscription_status
66,S0000067,C0000067,2014-01-01,NaT,UNKNOWN_STATE
74,S0000075,C0000075,2024-12-13,NaT,UNKNOWN_STATE
128,S0000129,C0000129,2014-01-01,2018-10-07,UNKNOWN_STATE
185,S0000186,C0000186,2021-01-05,NaT,UNKNOWN_STATE
188,S0000189,C0000189,2015-08-02,NaT,UNKNOWN_STATE
212,S0000213,C0000213,2021-10-19,NaT,UNKNOWN_STATE
905,S0000906,C0000906,2023-08-15,NaT,UNKNOWN_STATE
979,S0000980,C0000980,2017-08-15,NaT,UNKNOWN_STATE
1332,S0001333,C0001333,2025-06-01,NaT,UNKNOWN_STATE
1574,S0001575,C0001575,2015-07-23,NaT,UNKNOWN_STATE


In [66]:
subscriptions = clean_tables["subscriptions"]

# Preserve original statuses
if "original_subscription_status" not in subscriptions.columns:
    subscriptions["original_subscription_status"] = (
        subscriptions["subscription_status"]
    )

unknown_mask = (
    subscriptions["subscription_status"] == "UNKNOWN_STATE"
)

if "status_inferred" not in subscriptions.columns:
    subscriptions["status_inferred"] = False

subscriptions.loc[unknown_mask, "status_inferred"] = True

# Infer statuses using the observed end-date pattern
subscriptions.loc[
    unknown_mask & subscriptions["end_date"].isna(),
    "subscription_status"
] = "Active"

subscriptions.loc[
    unknown_mask & subscriptions["end_date"].notna(),
    "subscription_status"
] = "Terminated"

# Refresh the flag to reflect current values
subscriptions["invalid_subscription_status"] = (
    ~subscriptions["subscription_status"].isin(["Active", "Terminated"])
)

print("Statuses inferred this run:", unknown_mask.sum())
print(
    "Invalid statuses remaining:",
    subscriptions["invalid_subscription_status"].sum()
)

display(
    subscriptions["subscription_status"]
    .value_counts()
    .rename_axis("Subscription Status")
    .reset_index(name="Count")
)

Statuses inferred this run: 102
Invalid statuses remaining: 0


,Subscription Status,Count
0,Active,17017
1,Terminated,3506


In [67]:
valid_customer_ids = clean_tables["customers"]["customer_id"].dropna()

customer_fk_report = []

for table_name, df in clean_tables.items():
    if table_name != "customers" and "customer_id" in df.columns:
        missing_mask = df["customer_id"].isna()

        broken_mask = (
            df["customer_id"].notna()
            & ~df["customer_id"].isin(valid_customer_ids)
        )

        customer_fk_report.append({
            "Table": table_name,
            "Missing Customer IDs": missing_mask.sum(),
            "Nonexistent Customer IDs": broken_mask.sum()
        })

customer_fk_report = pd.DataFrame(customer_fk_report)

display(customer_fk_report)

,Table,Missing Customer IDs,Nonexistent Customer IDs
0,billing,0,0
1,complaints,0,0
2,contracts,0,0
3,customer_feedback,0,0
4,devices,0,0
5,network_quality,0,0
6,payments,0,0
7,plan_history,0,0
8,recharges,0,0
9,retention_campaigns,0,0


In [68]:
customers = clean_tables["customers"]

print("Customer column names:")
print(customers.columns.tolist())

# Preview fields relevant to the next quality checks
check_columns = [
    column for column in customers.columns
    if any(
        keyword in column.lower()
        for keyword in ["age", "birth", "phone", "mobile", "pin", "email"]
    )
]

display(customers[["customer_id"] + check_columns].head(10))

Customer column names:
['customer_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'age', 'email', 'phone_number', 'address', 'city_id', 'city_name', 'state_id', 'pin_code', 'city_tier', 'region_id', 'occupation', 'annual_income_inr', 'customer_segment', 'product_line', 'plan_id', 'acquisition_channel', 'acquisition_date', 'tenure_months', 'customer_status', 'churn_date']


,customer_id,date_of_birth,age,email,phone_number,pin_code
0,C0000001,1988-03-31,38,harish.shetty909@yahoo.in,916647481156,492680
1,C0000002,1974-03-31,52,anjali.khan525@rediffmail.com,918116536619,122535
2,C0000003,1978-03-31,48,arjun.hegde378@hotmail.com,918958153134,390978
3,C0000004,1988-03-31,38,vidya.parmar387@hotmail.com,917763920098,440881
4,C0000005,1988-03-31,38,arjun.saxena410@yahoo.in,916787478273,506582
5,C0000006,1990-03-31,36,akash.ansari961@hotmail.com,916083413392,796644
6,C0000007,2000-03-31,26,nikhil.shah569@yahoo.in,916529964485,769163
7,C0000008,1991-04-01,35,sanjay.hussain460@hotmail.com,918039201799,520224
8,C0000009,1996-03-31,30,harish.chatterjee164@rediffmail.com,918153754834,495460
9,C0000010,2006-03-31,20,imran.saha293@outlook.com,916422921624,560151


In [69]:
customers = clean_tables["customers"]

# Check numeric ages without changing the original column
age_values = pd.to_numeric(customers["age"], errors="coerce")

impossible_age_mask = (
    (age_values < 0) | (age_values > 120)
)

customers["invalid_age"] = impossible_age_mask

print("Missing ages:", customers["age"].isna().sum())
print(
    "Nonnumeric ages:",
    (customers["age"].notna() & age_values.isna()).sum()
)
print("Impossible ages:", impossible_age_mask.sum())

display(
    customers.loc[
        impossible_age_mask,
        ["customer_id", "age", "date_of_birth", "acquisition_date"]
    ].head(15)
)

Missing ages: 0
Nonnumeric ages: 0
Impossible ages: 23


,customer_id,age,date_of_birth,acquisition_date
751,C0000752,220,1989-03-31,2023-05-14
834,C0000835,220,1988-03-31,2018-07-26
2917,C0002918,130,1984-03-31,2014-01-01
3026,C0003027,220,1971-04-01,2023-09-14
3822,C0003823,220,1983-04-01,2014-01-01
3824,C0003825,220,1979-04-01,2022-03-26
4639,C0004640,130,1982-03-31,2023-08-08
5159,C0005160,130,1969-03-31,2020-04-10
5290,C0005291,130,2006-03-31,2014-01-01
5833,C0005834,-5,1982-03-31,2023-09-14


In [70]:
customers = clean_tables["customers"]

print("Age distribution — lowest and highest values:")
age_counts = customers["age"].value_counts().sort_index()

display(
    pd.concat([age_counts.head(10), age_counts.tail(10)])
    .loc[lambda s: ~s.index.duplicated()]
    .rename_axis("Age")
    .reset_index(name="Customer Count")
)

print("\nCustomers with ages from 0 to 17, or 100 to 120:")
unusual_age_mask = (
    customers["age"].between(0, 17)
    | customers["age"].between(100, 120)
)

display(
    customers.loc[
        unusual_age_mask,
        ["customer_id", "age", "date_of_birth"]
    ].head(30)
)

print("Rows in these age ranges:", unusual_age_mask.sum())

Age distribution — lowest and highest values:


,Age,Customer Count
0,-5,6
1,0,3
2,1,12
3,18,1456
4,19,231
5,20,295
6,21,328
7,22,335
8,23,371
9,24,378



Customers with ages from 0 to 17, or 100 to 120:


,customer_id,age,date_of_birth
506,C0000507,1,1992-03-31
868,C0000869,1,1990-03-31
1604,C0001605,0,2002-03-31
2175,C0002176,0,1979-04-01
4962,C0004963,1,1985-03-31
9266,C0009267,1,2002-03-31
9326,C0009327,1,2007-04-01
9841,C0009842,1,1993-03-31
13691,C0013692,1,1977-03-31
13780,C0013781,1,1996-03-31


Rows in these age ranges: 15


In [71]:
customers = clean_tables["customers"]

# Preserve original ages
if "original_age" not in customers.columns:
    customers["original_age"] = customers["age"]

original_age = pd.to_numeric(
    customers["original_age"], errors="coerce"
)

# Include the 0 and 1 values identified during inspection
invalid_age_mask = (
    (original_age < 0)
    | (original_age > 120)
    | original_age.isin([0, 1])
)

# Retain the flag as a record of the original issue
customers["invalid_age"] = invalid_age_mask

# Nullable integer type allows missing ages
customers["age"] = pd.to_numeric(
    customers["age"], errors="coerce"
).astype("Int64")

customers.loc[invalid_age_mask, "age"] = pd.NA

print("Incorrect ages flagged:", invalid_age_mask.sum())
print("Missing ages after cleaning:", customers["age"].isna().sum())

display(
    customers.loc[
        invalid_age_mask,
        ["customer_id", "original_age", "age", "date_of_birth"]
    ].head(10)
)

Incorrect ages flagged: 38
Missing ages after cleaning: 38


,customer_id,original_age,age,date_of_birth
506,C0000507,1,<NA>,1992-03-31
751,C0000752,220,<NA>,1989-03-31
834,C0000835,220,<NA>,1988-03-31
868,C0000869,1,<NA>,1990-03-31
1604,C0001605,0,<NA>,2002-03-31
2175,C0002176,0,<NA>,1979-04-01
2917,C0002918,130,<NA>,1984-03-31
3026,C0003027,220,<NA>,1971-04-01
3822,C0003823,220,<NA>,1983-04-01
3824,C0003825,220,<NA>,1979-04-01


In [72]:
customers = clean_tables["customers"]

# PHONE NUMBER CHECK
phone = customers["phone_number"].astype("string").str.strip()

missing_phone = phone.isna() | phone.eq("").fillna(False)

valid_phone = phone.str.fullmatch(
    r"(?:\+?91)?[6-9][0-9]{9}", na=False
)

customers["invalid_phone"] = ~missing_phone & ~valid_phone

# PIN CODE FORMAT CHECK: six digits, first digit cannot be zero
pin = customers["pin_code"].astype("string").str.strip()

missing_pin = pin.isna() | pin.eq("").fillna(False)

valid_pin = pin.str.fullmatch(r"[1-9][0-9]{5}", na=False)

customers["invalid_pin_format"] = ~missing_pin & ~valid_pin

# SUMMARY
display(pd.DataFrame({
    "Check": [
        "Missing phone numbers",
        "Invalid phone format",
        "Missing PIN codes",
        "Invalid PIN format"
    ],
    "Count": [
        missing_phone.sum(),
        customers["invalid_phone"].sum(),
        missing_pin.sum(),
        customers["invalid_pin_format"].sum()
    ]
}))

print("\nExamples of invalid phone numbers:")
display(
    customers.loc[
        customers["invalid_phone"],
        ["customer_id", "phone_number"]
    ].head(10)
)

print("\nExamples of invalid PIN formats:")
display(
    customers.loc[
        customers["invalid_pin_format"],
        ["customer_id", "pin_code", "city_name"]
    ].head(10)
)

,Check,Count
0,Missing phone numbers,0
1,Invalid phone format,228
2,Missing PIN codes,0
3,Invalid PIN format,190



Examples of invalid phone numbers:


,customer_id,phone_number
10,C0000011,57059
65,C0000066,13624
138,C0000139,91566
404,C0000405,75835
405,C0000406,55071
524,C0000525,70233
544,C0000545,84892
579,C0000580,24255
585,C0000586,97577
663,C0000664,47167



Examples of invalid PIN formats:


,customer_id,pin_code,city_name
45,C0000046,9172,Chandigarh
182,C0000183,7123,Lucknow
185,C0000186,7128,Durgapur
360,C0000361,4841,Dehradun
517,C0000518,8109,Ludhiana
534,C0000535,2180,Bhagalpur
586,C0000587,1470,Chennai
610,C0000611,5514,Lucknow
673,C0000674,9869,Noida
679,C0000680,6401,Noida


In [73]:
complaints = clean_tables["complaints"]

# Standardise labels temporarily for counting
status = (
    complaints["status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

print("Complaint status counts:")
display(complaints["status"].value_counts(dropna=False))

unresolved_count = status.eq("unresolved").fillna(False).sum()
total_complaints = len(complaints)

unresolved_pct = (
    unresolved_count / total_complaints * 100
    if total_complaints else 0
)

print("\nTotal complaints:", total_complaints)
print("Unresolved complaints:", unresolved_count)
print(f"Unresolved complaints percentage: {unresolved_pct:.2f}%")

Complaint status counts:


status
Resolved      9236
Unresolved    7348
Name: count, dtype: int64


Total complaints: 16584
Unresolved complaints: 7348
Unresolved complaints percentage: 44.31%


In [74]:
complaints = clean_tables["complaints"]

status = (
    complaints["status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

display(
    pd.crosstab(
        status,
        complaints["resolution_date"].notna()
    ).rename(columns={
        False: "Missing Resolution Date",
        True: "Has Resolution Date"
    })
)

print(
    "Resolved complaints missing a resolution date:",
    (status.eq("resolved") & complaints["resolution_date"].isna()).sum()
)

print(
    "Unresolved complaints with a resolution date:",
    (status.eq("unresolved") & complaints["resolution_date"].notna()).sum()
)

print(
    "Resolution dates before complaint dates:",
    (
        complaints["resolution_date"]
        < complaints["complaint_date"]
    ).sum()
)

resolution_date,Missing Resolution Date,Has Resolution Date
status,,
resolved,0,9236
unresolved,7348,0


Resolved complaints missing a resolution date: 0
Unresolved complaints with a resolution date: 0
Resolution dates before complaint dates: 0


In [75]:
customers = clean_tables["customers"]

email = customers["email"].astype("string").str.strip()

missing_email = email.isna() | email.eq("").fillna(False)

# Basic format check: name@domain.extension, without spaces
valid_email = email.str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+",
    na=False
)

customers["invalid_email_format"] = ~missing_email & ~valid_email

print("Missing or blank emails:", missing_email.sum())
print("Invalid email formats:", customers["invalid_email_format"].sum())

display(
    customers.loc[
        customers["invalid_email_format"],
        ["customer_id", "email"]
    ].head(10)
)

Missing or blank emails: 744
Invalid email formats: 285


,customer_id,email
203,C0000204,user544_at_mail
268,C0000269,user123_at_mail
481,C0000482,user615_at_mail
717,C0000718,user428_at_mail
822,C0000823,user690_at_mail
829,C0000830,user648_at_mail
830,C0000831,user521_at_mail
857,C0000858,user564_at_mail
869,C0000870,user731_at_mail
998,C0000999,user902_at_mail


In [76]:
cities = clean_tables["cities"]

print("City reference columns:")
print(cities.columns.tolist())

display(cities.head(5))

print("\nMissing city IDs in reference table:",
      cities["city_id"].isna().sum())

print("Duplicate city IDs in reference table:",
      cities["city_id"].duplicated().sum())

print("Customer city IDs absent from reference table:",
      (
          customers["city_id"].notna()
          & ~customers["city_id"].isin(cities["city_id"])
      ).sum())

City reference columns:
['city_id', 'city_name', 'state_id', 'state_name', 'zone', 'region_id', 'tier', 'pin_prefix']


,city_id,city_name,state_id,state_name,zone,region_id,tier,pin_prefix
0,CT0001,New Delhi,ST001,Delhi,North,RG01,Metro,110
1,CT0002,Mumbai,ST010,Maharashtra,West,RG02,Metro,400
2,CT0003,Bengaluru,ST015,Karnataka,South,RG03,Metro,560
3,CT0004,Chennai,ST016,Tamil Nadu,South,RG03,Metro,600
4,CT0005,Kolkata,ST022,West Bengal,East,RG04,Metro,700



Missing city IDs in reference table: 0
Duplicate city IDs in reference table: 0
Customer city IDs absent from reference table: 0


In [77]:
customers = clean_tables["customers"]
cities = clean_tables["cities"]

city_lookup = cities.set_index("city_id")["city_name"]

city_check = customers[
    ["customer_id", "city_id", "city_name"]
].copy()

city_check["reference_city_name"] = (
    city_check["city_id"].map(city_lookup)
)

actual = city_check["city_name"].astype("string")
expected = city_check["reference_city_name"].astype("string")

comparable = actual.notna() & expected.notna()

exact_match = actual.eq(expected).fillna(False)
normalised_match = (
    actual.str.strip().str.casefold()
    .eq(expected.str.strip().str.casefold())
    .fillna(False)
)

format_mask = comparable & ~exact_match & normalised_match
name_mask = comparable & ~normalised_match

print("Capitalisation or spacing differences:", format_mask.sum())
print("Spelling or other name differences:", name_mask.sum())
print("Missing customer city names:", actual.isna().sum())

print("\nExamples of differences:")
display(
    city_check.loc[format_mask | name_mask]
    .drop_duplicates(["city_name", "reference_city_name"])
    .head(20)
)

Capitalisation or spacing differences: 560
Spelling or other name differences: 380
Missing customer city names: 0

Examples of differences:


,customer_id,city_id,city_name,reference_city_name
12,C0000013,CT0020,VADODARA,Vadodara
33,C0000034,CT0006,Hyderaabad,Hyderabad
53,C0000054,CT0008,Ahmedaabad,Ahmedabad
66,C0000067,CT0006,HYDERABAD,Hyderabad
79,C0000080,CT0065,SHILLONG,Shillong
84,C0000085,CT0007,PUNE,Pune
93,C0000094,CT0041,Belaagavi,Belagavi
97,C0000098,CT0052,SILIGURI,Siliguri
138,C0000139,CT0003,BENGALURU,Bengaluru
160,C0000161,CT0036,Kaanpur,Kanpur


In [78]:
customers = clean_tables["customers"]
cities = clean_tables["cities"]

# Preserve original names
if "original_city_name" not in customers.columns:
    customers["original_city_name"] = customers["city_name"]

# Find the reference name for each customer's city ID
reference_city = customers["city_id"].map(
    cities.set_index("city_id")["city_name"]
)

city_fix_mask = (
    reference_city.notna()
    & customers["city_name"].ne(reference_city).fillna(True)
)

# Standardise names using the reference table
customers.loc[city_fix_mask, "city_name"] = reference_city[city_fix_mask]

# Retain an audit flag
customers["city_name_corrected"] = (
    reference_city.notna()
    & customers["original_city_name"].ne(reference_city).fillna(True)
)

print("City names corrected this run:", city_fix_mask.sum())

print(
    "Remaining city-name mismatches:",
    (
        reference_city.notna()
        & customers["city_name"].ne(reference_city).fillna(True)
    ).sum()
)

display(
    customers.loc[
        customers["city_name_corrected"],
        ["customer_id", "city_id", "original_city_name", "city_name"]
    ].head(10)
)

City names corrected this run: 940
Remaining city-name mismatches: 0


,customer_id,city_id,original_city_name,city_name
12,C0000013,CT0020,VADODARA,Vadodara
33,C0000034,CT0006,Hyderaabad,Hyderabad
53,C0000054,CT0008,Ahmedaabad,Ahmedabad
66,C0000067,CT0006,HYDERABAD,Hyderabad
79,C0000080,CT0065,SHILLONG,Shillong
84,C0000085,CT0007,PUNE,Pune
93,C0000094,CT0041,Belaagavi,Belagavi
97,C0000098,CT0052,SILIGURI,Siliguri
138,C0000139,CT0003,BENGALURU,Bengaluru
160,C0000161,CT0036,Kaanpur,Kanpur


In [79]:
usage = clean_tables["usage_data"]

q1 = usage["data_gb_used"].quantile(0.25)
q3 = usage["data_gb_used"].quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

usage["usage_outlier_iqr"] = (
    (usage["data_gb_used"] < lower_limit)
    | (usage["data_gb_used"] > upper_limit)
)

print("Lower IQR limit:", round(lower_limit, 2))
print("Upper IQR limit:", round(upper_limit, 2))
print("Usage rows flagged:", usage["usage_outlier_iqr"].sum())
print("Rows retained:", len(usage))

display(
    usage.loc[usage["usage_outlier_iqr"]]
    .sort_values("data_gb_used", ascending=False)
    .head(10)
)

Lower IQR limit: -20.47
Upper IQR limit: 64.49
Usage rows flagged: 4480
Rows retained: 57000


,usage_id,customer_id,usage_month,data_gb_used,data_sessions,usage_outlier_iqr
15077,UD00015078,C0005026,01-2026,22172.700708,31,True
23089,UD00023090,C0007697,02-2026,21960.635745,489,True
10054,UD00010055,C0003352,02-2026,18348.446462,369,True
34859,UD00034860,C0011620,01-2026,17377.228312,765,True
26592,UD00026593,C0008865,03-2026,15322.701632,268,True
29837,UD00029838,C0009946,01-2026,13886.727541,1105,True
53991,UD00053992,C0017998,03-2026,13344.522727,862,True
56096,UD00056097,C0018699,01-2026,13154.906387,317,True
45208,UD00045209,C0015070,02-2026,12747.455634,929,True
19929,UD00019930,C0006644,03-2026,12581.380386,496,True


In [80]:
missing_summary = []

for name, df in clean_tables.items():
    for column in tables[name].columns:
        values = df[column]

        # Count both nulls and blank strings
        missing = (
            values.isna()
            | values.astype("string").str.strip().eq("").fillna(False)
        )

        count = int(missing.sum())

        if count > 0:
            missing_summary.append({
                "Table": name,
                "Column": column,
                "Missing Count": count,
                "Missing %": round(count / len(df) * 100, 2)
            })

final_missing_report = pd.DataFrame(
    missing_summary,
    columns=["Table", "Column", "Missing Count", "Missing %"]
).sort_values(["Table", "Column"]).reset_index(drop=True)

display(final_missing_report)

,Table,Column,Missing Count,Missing %
0,complaints,resolution_date,7348,44.31
1,customers,address,380,2.00
2,customers,age,38,0.20
3,customers,annual_income_inr,950,5.00
4,customers,churn_date,15494,81.55
5,customers,email,744,3.92
6,subscriptions,end_date,17017,82.92
7,subscriptions,plan_id,41,0.20
8,support_tickets,resolution_date,13366,42.45
9,support_tickets,resolution_hours,13366,42.45


In [81]:
# Customer status versus missing churn date
print("Customer status and churn dates:")
display(
    pd.crosstab(
        customers["customer_status"].fillna("Missing status"),
        customers["churn_date"].isna().map({
            True: "Missing churn date",
            False: "Has churn date"
        })
    )
)

# Identify the ticket status column
tickets = clean_tables["support_tickets"]
status_columns = [
    col for col in tickets.columns
    if "status" in col.lower()
]

print("Ticket status columns:", status_columns)

for col in status_columns:
    print(f"\n{col} versus missing resolution date:")
    display(
        pd.crosstab(
            tickets[col].fillna("Missing status"),
            tickets["resolution_date"].isna().map({
                True: "Missing resolution date",
                False: "Has resolution date"
            })
        )
    )

Customer status and churn dates:


churn_date,Has churn date,Missing churn date
customer_status,,
Active,0,15494
Churned,3506,0


Ticket status columns: ['status']

status versus missing resolution date:


resolution_date,Has resolution date,Missing resolution date
status,,
Escalated,0,2526
Pending,0,8450
Reopened,0,2390
Resolved,18124,0


In [82]:
# Build the comparison from the checks already performed above.
# Run the notebook from the beginning before running this cell.

customers_qc = clean_tables["customers"]
subscriptions_qc = clean_tables["subscriptions"]


# Count null values and blank strings
def count_missing(series):
    missing = (
        series.isna()
        | series.astype("string").str.strip().eq("").fillna(False)
    )
    return int(missing.sum())


# Check original billing dates, before date corrections.
# This remains an inferred dataset cutoff.
reference_date = pd.Timestamp("2026-03-31")

original_billing_qc = tables["billing"].drop_duplicates()

original_dates_qc = pd.to_datetime(
    original_billing_qc["billing_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

future_count = int(
    (original_dates_qc > reference_date).sum()
)


# Calculate observed counts automatically
observed_counts = {
    ("customers", "missing_email"):
        count_missing(customers_qc["email"]),

    ("customers", "missing_address"):
        count_missing(customers_qc["address"]),

    ("customers", "missing_income"):
        count_missing(customers_qc["annual_income_inr"]),

    ("customers", "invalid_phone"):
        int(customers_qc["invalid_phone"].sum()),

    ("customers", "invalid_email"):
        int(customers_qc["invalid_email_format"].sum()),

    ("customers", "incorrect_pin"):
        int(customers_qc["invalid_pin_format"].sum()),

    ("customers", "misspelled_city"):
        int(name_mask.sum()),

    ("customers", "inconsistent_capitalisation"):
        int(format_mask.sum()),

    # Uses your existing broader age-quality flag.
    # This includes out-of-range ages and the flagged ages 0/1.
    ("customers", "impossible_age"):
        int(customers_qc["invalid_age"].sum()),

    ("customers", "mixed_date_format"):
        int(len(recovered)),

    ("customers", "duplicate_customers"):
        int(tables["customers"].duplicated().sum()),

    ("billing", "incorrect_gst"):
        int(len(gst_errors)),

    ("billing", "negative_invoice_amount"):
        int(len(negative_bills)),

    ("billing", "duplicate_invoices"):
        int(tables["billing"].duplicated().sum()),

    ("billing", "future_dates"):
        future_count,

    ("payments", "broken_foreign_key"):
        int(broken_invoice_mask.sum()),

    ("payments", "duplicate_payments"):
        int(tables["payments"].duplicated().sum()),

    ("subscriptions", "null_foreign_key"):
        int(subscriptions_qc["plan_id"].isna().sum()),

    ("subscriptions", "invalid_category"):
           int(
        (
            ~tables["subscriptions"]
            .drop_duplicates()["subscription_status"]
            .isin(["Active", "Terminated"])
        ).sum()
    ),

    ("usage_data", "outlier_values"):
        int(clean_tables["usage_data"]["usage_outlier_iqr"].sum())
}


# Compare with the supplied issue log
issue_comparison = tables["data_quality_issue_log"].copy()

issue_comparison["Observed Count"] = [
    observed_counts.get((table, issue), pd.NA)
    for table, issue in zip(
        issue_comparison["table"],
        issue_comparison["issue_type"]
    )
]

issue_comparison["Observed Count"] = (
    issue_comparison["Observed Count"].astype("Int64")
)

display(issue_comparison)

,table,issue_type,rows_affected,Observed Count
0,customers,missing_email,760,744
1,customers,missing_address,380,380
2,customers,missing_income,950,950
3,customers,invalid_phone,228,228
4,customers,invalid_email,285,285
5,customers,incorrect_pin,190,190
6,customers,misspelled_city,380,380
7,customers,inconsistent_capitalisation,570,560
8,customers,impossible_age,38,38
9,customers,mixed_date_format,380,380


In [83]:
comparison_notes = {
    ("customers", "missing_email"):
        "744 missing after deduplication; log lists 760. Difference unresolved.",
    ("customers", "inconsistent_capitalisation"):
        "560 city names differ only by case/spacing; log lists 570.",
    ("customers", "impossible_age"):
        "23 ages below 0 or above 120, plus 15 ages inconsistent with DOB.",
    ("billing", "future_dates"):
        "Dataset cutoff unconfirmed. Separately, 229 billing dates were "
        "after due dates; replacements inferred as due date minus 15 days.",
    ("usage_data", "outlier_values"):
        "IQR rule flags 4480 potential outliers; log lists 171 errors. "
        "Different definitions; all rows retained."
}

issue_comparison["Notes"] = [
    comparison_notes.get(
        (table, issue),
        "Observed count matches the issue log."
    )
    for table, issue in zip(
        issue_comparison["table"],
        issue_comparison["issue_type"]
    )
]

display(issue_comparison)

,table,issue_type,rows_affected,Observed Count,Notes
0,customers,missing_email,760,744,744 missing after deduplication; log lists 760...
1,customers,missing_address,380,380,Observed count matches the issue log.
2,customers,missing_income,950,950,Observed count matches the issue log.
3,customers,invalid_phone,228,228,Observed count matches the issue log.
4,customers,invalid_email,285,285,Observed count matches the issue log.
5,customers,incorrect_pin,190,190,Observed count matches the issue log.
6,customers,misspelled_city,380,380,Observed count matches the issue log.
7,customers,inconsistent_capitalisation,570,560,560 city names differ only by case/spacing; lo...
8,customers,impossible_age,38,38,"23 ages below 0 or above 120, plus 15 ages inc..."
9,customers,mixed_date_format,380,380,Observed count matches the issue log.


In [86]:
from pathlib import Path

output_folder = Path("phase1_output")
clean_folder = output_folder / "cleaned_data"
report_folder = output_folder / "quality_reports"

clean_folder.mkdir(parents=True, exist_ok=True)
report_folder.mkdir(parents=True, exist_ok=True)

for name, df in clean_tables.items():
    df.to_csv(
        clean_folder / f"{name}.csv",
        index=False,
        date_format="%Y-%m-%d"
    )

final_missing_report.to_csv(
    report_folder / "missing_values_report.csv",
    index=False
)

issue_comparison.to_csv(
    report_folder / "issue_log_comparison.csv",
    index=False
)

customer_fk_report.to_csv(
    report_folder / "customer_foreign_key_report.csv",
    index=False
)

print("Tables exported:", len(clean_tables))
print("Output location:", output_folder.resolve())
print("Quality reports saved: 3")

Tables exported: 24
Output location: C:\Users\Anushka Tyagi\NEXATEL_PROJECT\phase1_output
Quality reports saved: 3


In [87]:
# Inferred dataset cutoff, not an officially supplied snapshot date
reference_date = pd.Timestamp("2026-03-31")

original_billing = tables["billing"].drop_duplicates().copy()

original_dates = pd.to_datetime(
    original_billing["billing_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

future_mask = original_dates > reference_date
future_count = int(future_mask.sum())

print("Inferred reference date:", reference_date.date())
print("Original invoices after reference date:", future_count)

row_mask = (
    issue_comparison["table"].eq("billing")
    & issue_comparison["issue_type"].eq("future_dates")
)

issue_comparison.loc[row_mask, "Observed Count"] = future_count
issue_comparison.loc[row_mask, "Notes"] = (
    "229 billing dates after inferred cutoff 2026-03-31. "
    "Cutoff supported by billing patterns; not explicitly documented. "
    "Replacement dates inferred from due date minus 15 days."
)

issue_comparison.to_csv(
    report_folder / "issue_log_comparison.csv",
    index=False
)

Inferred reference date: 2026-03-31
Original invoices after reference date: 229


# Phase 1 — Data Cleaning & Quality Report
**Project:** NexaTel Customer Churn Analytics  
**Prepared by:** Anushka Tyagi

**Scope and approach.** Profiled all 24 CSV files for row counts, column counts, data types, missing values, duplicates and referential integrity. The original files contained 756,952 rows. Removing 696 exact duplicates—76 customer, 344 billing and 276 payment rows—left 756,256 rows. Customer and invoice IDs were unique and non-null after deduplication. Original input files were preserved.

**Data types and corrections.** Converted 23 date columns to datetime, including recovery of 380 customer acquisition dates stored as YYYY/MM/DD instead of DD-MM-YYYY. Inspected monetary and usage fields for numeric types. Corrected 1,149 GST discrepancies using the 18% rate specified in the project instructions and corrected 229 negative invoice totals. Validation found no remaining negative totals, GST discrepancies or differences between invoice totals and base amount plus GST within the applied tolerance. Standardised 940 city names using the city reference table: 560 case/spacing differences and 380 spelling differences.

**Date and status assumptions.** Using **31 March 2026 as an inferred dataset cutoff**, identified 229 original invoices with future billing dates, matching the issue-log count. These 229 invoices were also dated after their due dates. Their billing dates were reconstructed as due date minus 15 days, following the pattern in the other 114,740 invoices. The cutoff and reconstructed dates are inferred, not explicitly verified by source documentation. Similarly, 102 invalid subscription statuses were inferred from end-date presence: 87 became Active and 15 became Terminated. Original values and audit flags were retained.

**Customer quality and missing values.** Identified **23 impossible ages** below 0 or above 120, plus **15 additional ages inconsistent with birth dates**. All 38 were replaced with missing values while preserving originals. Flagged 228 invalid phone formats, 190 invalid PIN formats and 285 invalid email formats. These checks validate format, not whether contact details are operational. Retained 744 missing emails (3.92%), 380 addresses (2.00%), 950 incomes (5.00%) and 41 subscription plan IDs (0.20%); replacements were not guessed.

**Integrity, outliers and service checks.** Customer references across 15 checked tables had no missing or nonexistent customer IDs. However, 184 payments referenced nonexistent invoices; these were retained and flagged. No non-null subscription plan IDs referenced nonexistent plans. The 1.5×IQR rule flagged 4,480 of 57,000 data-usage rows (7.86%), with an upper threshold of approximately 64.49 GB. All usage rows were retained because statistical outliers are not automatically errors. Unresolved complaints numbered 7,348 of 16,584 (44.31%). Missing complaint resolution dates, customer churn dates and ticket resolution dates were consistent with their respective statuses.

**Issue-log comparison and conclusion.** Observed missing emails differed from the log (744 versus 760), as did city formatting differences (560 versus 570); the causes remain unconfirmed. The 4,480 IQR flags are not directly comparable with the log’s 171 injected outlier errors. The age total of 38 includes two distinct checks, and matching counts alone does not establish row-level agreement. Exported 24 tables and three quality reports covering missing values, issue-log comparison and customer foreign keys. Subsequent analysis should account for retained missing values, broken references, potential outliers and documented inference assumptions.